# Figuras de la tesis (Figura 2 y Figuras 4 a 9)

José Andrés Estrada Morales · Carné 1031722

Genera las figuras del planteamiento del problema (Figura 2) y del capítulo de resultados (Figuras 4 a 9) en el formato
de la tesis. Usa la base de datos y las salidas de `codigo_tesis_v1` y `nardl_complementario`, así que primero hay que
correr esos dos notebooks. Debe estar en la misma carpeta que ellos y que Base_datos_tesis.xlsx.

In [ ]:
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
plt.rcParams.update({'font.family': 'serif', 'font.serif': ['Liberation Serif', 'Times New Roman', 'DejaVu Serif'],
                     'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})
BASE = Path('Base_datos_tesis.xlsx')
R = Path('resultados_v1_saldo_normalizado'); N = Path('resultados_nardl_saldo_normalizado'); O = Path('figuras_tesis')
O.mkdir(exist_ok=True)
t = pd.read_csv(R / '09_datos_transformados.csv', index_col=0, parse_dates=True)
AZ, GR, RO, VE, NA = '#1F4E79', '#7F7F7F', '#B5403C', '#2E7D32', '#C9A227'
P = [('cafe', 'Café', 'US$/kg'), ('azucar', 'Azúcar', 'US$/kg'), ('cardamomo', 'Cardamomo (valor unitario)', 'US$/kg'),
     ('banano', 'Banano', 'US$/kg'), ('petroleo', 'Petróleo WTI', 'US$/barril')]

# Figura 2: precios internacionales, índice 2010 = 100 (promedios anuales)
b = pd.read_excel(BASE, sheet_name='BASE_CORREGIDA'); b['Periodo'] = pd.to_datetime(b['Periodo']); b = b.set_index('Periodo')
S2 = [('Café Arábica (US$/kg)', 'Café', AZ, 'o'), ('Azúcar mundial (US$/kg)', 'Azúcar', GR, 's'),
      ('Cardamomo: valor unitario exportación (US$/kg)', 'Cardamomo (valor unitario)', VE, '^'),
      ('Banano EE.UU. (US$/kg)', 'Banano', NA, 'D'), ('Petróleo WTI (US$/bbl)', 'Petróleo WTI', RO, 'v')]
fig, ax = plt.subplots(figsize=(10, 5.3))
for c, l, col, mk in S2:
    a_ = b[c].resample('YE').mean(); ax.plot(a_.index.year, 100 * a_ / a_.iloc[0], color=col, marker=mk, lw=2, ms=6, label=l)
ax.axhline(100, color='k', lw=.8); ax.set_ylabel('Índice (2010 = 100)')
ax.set_xticks(range(2010, 2025)); ax.set_xticklabels(range(2010, 2025), rotation=45); ax.grid(axis='y', color='#e5e5e5')
ax.legend(ncol=3, frameon=False, loc='upper center', bbox_to_anchor=(0.5, 1.13))
fig.tight_layout(); fig.savefig(O / 'fig2.png', dpi=200); plt.close(fig)

# Figura 4: saldo mensual en US$ y normalizado
fig, ax = plt.subplots(2, 1, figsize=(10, 6.6), sharex=True)
ax[0].plot(t.index, t.balanza, color=AZ, lw=1.4); ax[0].axhline(0, color='k', lw=.7)
ax[0].set_ylabel('Millones de US$'); ax[0].set_title('a) Saldo comercial mensual (US$ millones)', loc='left', fontsize=11)
ax[1].plot(t.index, t.saldo_norm, color=RO, lw=1.4)
ax[1].plot(t.saldo_norm.rolling(12).mean(), color='k', lw=1, ls='--', label='Promedio móvil de 12 meses')
ax[1].set_ylabel('% del comercio total'); ax[1].legend(frameon=False, loc='lower left')
ax[1].set_title('b) Saldo comercial normalizado, 100 × (X − M) / (X + M)', loc='left', fontsize=11)
for a in ax: a.grid(axis='y', color='#e5e5e5')
fig.tight_layout(); fig.savefig(O / 'fig4.png', dpi=200); plt.close(fig)

# Figura 5: precios y retornos
fig, ax = plt.subplots(5, 2, figsize=(10, 11.5), sharex=True)
for i, (c, n, u) in enumerate(P):
    ax[i, 0].plot(t.index, t[c], color=AZ, lw=1.2); ax[i, 0].set_ylabel(u); ax[i, 0].set_title(f'{n}: precio', loc='left', fontsize=10)
    ax[i, 1].bar(t.index, t[f'r_{c}'], width=25, color=np.where(t[f'r_{c}'] >= 0, AZ, RO))
    ax[i, 1].axhline(0, color='k', lw=.6); ax[i, 1].set_ylabel('%'); ax[i, 1].set_title(f'{n}: retorno mensual', loc='left', fontsize=10)
fig.tight_layout(); fig.savefig(O / 'fig5.png', dpi=200); plt.close(fig)

# Figura 6: volatilidad condicional y desviación móvil
sel = pd.read_csv(R / '01_volatilidad_seleccion_y_diagnosticos.csv').set_index('serie')
fig, ax = plt.subplots(3, 2, figsize=(10, 9.5))
for a, (c, n, u) in zip(ax.flat, P):
    m = sel.loc[c]; et = f"{m.varianza} ({'t' if m.distribucion == 't' else m.distribucion})"
    a.plot(t.index, t[f'vol_{c}'], color=AZ, lw=1.5, label=et)
    a.plot(t.index, t[f'volm_{c}'], color=NA, lw=1, label='Desv. estándar móvil 12 meses')
    a.set_title(n, loc='left', fontsize=10); a.set_ylabel('σ mensual (%)'); a.legend(frameon=False, fontsize=8, loc='upper left')
ax.flat[-1].axis('off'); fig.tight_layout(); fig.savefig(O / 'fig6.png', dpi=200); plt.close(fig)

# Figura 7: CUSUM y CUSUMSQ
cs = pd.read_csv(R / '06_cusum_cusumsq_principal.csv')
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
for a, c, tt in [(ax[0], 'CUSUM', 'a) CUSUM'), (ax[1], 'CUSUMSQ', 'b) CUSUMSQ')]:
    a.plot(cs[c].values, color=AZ, lw=1.4); a.plot(cs[f'{c}_inf'].values, color=RO, ls='--', lw=1); a.plot(cs[f'{c}_sup'].values, color=RO, ls='--', lw=1)
    a.set_title(tt + ' (bandas al 5%)', loc='left', fontsize=10); a.set_xlabel('Observación recursiva')
fig.tight_layout(); fig.savefig(O / 'fig7.png', dpi=200); plt.close(fig)

# Figura 8: Toda–Yamamoto
ty = pd.read_csv(R / '08_toda_yamamoto.csv'); ty = ty[ty.dmax_principal]
nom = dict(vol_cafe='Café', vol_azucar='Azúcar', vol_cardamomo='Cardamomo', vol_banano='Banano', vol_petroleo='Petróleo WTI')
fig, a = plt.subplots(figsize=(10, 4.2)); x = np.arange(5)
for j, (cov, lab, col) in enumerate([('conv', 'Convencional', AZ), ('HC3', 'HC3', NA)]):
    g = ty[ty.covarianza == cov].set_index('variable').loc[list(nom)]
    b = a.bar(x + (j - .5) * .36, g.p_Holm, .36, color=col, label=lab)
    for bb, v in zip(b, g.p_Holm): a.text(bb.get_x() + bb.get_width() / 2, v + .02, '< 0.001' if v < .001 else f'{v:.3f}', ha='center', fontsize=8)
a.axhline(.05, color=RO, ls='--', lw=1, label='5%'); a.set_xticks(x, list(nom.values())); a.set_ylim(0, 1.3)
a.set_ylabel('p-valor ajustado por Holm'); a.legend(frameon=False, ncol=3, loc='upper center')
fig.tight_layout(); fig.savefig(O / 'fig8.png', dpi=200); plt.close(fig)

# Figura 9: NARDL
nd = pd.read_csv(N / 'nardl_resumen.csv')
fig, a = plt.subplots(figsize=(10, 4.4)); x = np.arange(len(nd))
for j, (s, lab, col) in enumerate([('pos', 'Alzas acumuladas (L+)', AZ), ('neg', 'Bajas acumuladas (L−)', RO)]):
    a.errorbar(x + (j - .5) * .22, nd[f'L_{s}'], yerr=1.96 * nd[f'L_{s}_ee'], fmt='o', capsize=4, color=col, label=lab)
a.axhline(0, color='k', lw=.7); a.set_xticks(x, nd.producto)
a.set_ylabel('Multiplicador de largo plazo\n(pp del saldo normalizado por 1%)'); a.legend(frameon=False, loc='lower left')
fig.tight_layout(); fig.savefig(O / 'fig9.png', dpi=200); plt.close(fig)
print('Figuras guardadas en', O.resolve())